# GNSS IV Lab: Fault Locking, Creep, and the Earthquake Slip Budget

In this lab you will transform GNSS velocities into San Andreas fault coordinates and fit a simple interseismic locking model.

**Learning goals**

- rotate positions and velocities into fault-parallel and fault-perpendicular components
- explain how slip rate and locking depth affect an interseismic velocity profile
- estimate model parameters from GNSS velocities
- distinguish broad elastic strain accumulation from localized creep
- convert slip deficit into an earthquake moment budget without treating it as a prediction

Data: [Kreemer et al. (2022) velocity archive](https://doi.org/10.7910/DVN/BICMWB)  
Direct file: [Harvard Dataverse file 6282416](https://dataverse.harvard.edu/file.xhtml?fileId=6282416&version=1.0)


## 1. Setup

The notebook first looks for a local course copy of the Kreemer velocity table. If none is present, it attempts to download the file from Harvard Dataverse.


In [ ]:
import io
import re
import zipfile
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA_URL = "https://dataverse.harvard.edu/api/access/datafile/6282416"
LOCAL_CANDIDATES = [
    Path("../data/kreemer_2022_gps_velocities.txt"),
    Path("data/kreemer_2022_gps_velocities.txt"),
    Path("kreemer_2022_gps_velocities.txt"),
]


## 2. Load and standardize the velocity table

The archive may use names such as `Ve`, `Evel`, or `east_velocity` for the same quantity. The next cells parse the table and map its columns onto a common set of names.


In [ ]:
def read_velocity_bytes(raw):
    if raw[:2] == b"PK":
        with zipfile.ZipFile(io.BytesIO(raw)) as archive:
            members = [n for n in archive.namelist() if not n.endswith("/")]
            preferred = [n for n in members if n.lower().endswith((".txt", ".csv", ".tsv", ".dat"))]
            member = (preferred or members)[0]
            raw = archive.read(member)
            print(f"Reading {member} from downloaded archive")

    for kwargs in [
        dict(sep=None, engine="python", comment="#"),
        dict(sep=r"\s+", engine="python", comment="#"),
    ]:
        try:
            table = pd.read_csv(io.BytesIO(raw), **kwargs)
            if table.shape[1] >= 4:
                return table
        except Exception:
            pass
    raise ValueError("The file could not be parsed as a velocity table.")


local_file = next((p for p in LOCAL_CANDIDATES if p.exists()), None)
if local_file is not None:
    print(f"Loading local data: {local_file}")
    raw = local_file.read_bytes()
else:
    print("No local copy found; downloading from Harvard Dataverse...")
    try:
        with urlopen(DATA_URL, timeout=60) as response:
            raw = response.read()
    except Exception as error:
        raise RuntimeError(
            "Automatic download failed. Download Dataverse file 6282416 in a browser, "
            "save it as data/kreemer_2022_gps_velocities.txt, and rerun this cell."
        ) from error

velocity_raw = read_velocity_bytes(raw)
print(f"Rows: {len(velocity_raw):,}; columns: {velocity_raw.shape[1]}")
display(velocity_raw.head())
print("Column names:", list(velocity_raw.columns))


In [ ]:
COLUMN_MAP = {
    "station": None,
    "lon": None,
    "lat": None,
    "ve": None,
    "vn": None,
    "se": None,
    "sn": None,
}

ALIASES = {
    "station": ["station", "site", "sta", "name", "code", "id"],
    "lon": ["longitude", "lon", "long"],
    "lat": ["latitude", "lat"],
    "ve": ["ve", "v_e", "eastvelocity", "east_vel", "evel", "vel_e", "east"],
    "vn": ["vn", "v_n", "northvelocity", "north_vel", "nvel", "vel_n", "north"],
    "se": ["se", "sve", "sig_e", "sigma_e", "east_sigma", "east_unc", "e_unc"],
    "sn": ["sn", "svn", "sig_n", "sigma_n", "north_sigma", "north_unc", "n_unc"],
}


def canonical(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


def infer_column(columns, aliases):
    normalized = {canonical(c): c for c in columns}
    for alias in aliases:
        if canonical(alias) in normalized:
            return normalized[canonical(alias)]
    for alias in aliases:
        matches = [original for key, original in normalized.items()
                   if key.startswith(canonical(alias))]
        if len(matches) == 1:
            return matches[0]
    return None


for key in COLUMN_MAP:
    if COLUMN_MAP[key] is None:
        COLUMN_MAP[key] = infer_column(velocity_raw.columns, ALIASES[key])

print("Detected column mapping:")
for key, value in COLUMN_MAP.items():
    print(f"  {key:>7s} <- {value}")

missing = [key for key, value in COLUMN_MAP.items() if value is None]
if missing:
    raise KeyError(f"Could not identify {missing}. Edit COLUMN_MAP using the printed columns.")

velocity = velocity_raw[[COLUMN_MAP[k] for k in COLUMN_MAP]].copy()
velocity.columns = list(COLUMN_MAP)
for column in ["lon", "lat", "ve", "vn", "se", "sn"]:
    velocity[column] = pd.to_numeric(velocity[column], errors="coerce")
velocity = velocity.dropna(subset=["lon", "lat", "ve", "vn", "se", "sn"]).copy()
velocity["station"] = velocity.station.astype(str)

VELOCITY_SCALE_TO_MM_PER_YR = 1.0  # Change if the archive uses m/yr.
for column in ["ve", "vn", "se", "sn"]:
    velocity[column] *= VELOCITY_SCALE_TO_MM_PER_YR

display(velocity.head())
print(f"Usable velocities: {len(velocity):,}")
print(velocity[["ve", "vn"]].describe())


### Data check

Confirm from the archive metadata that:

1. `ve` and `vn` are the velocities used in the analysis, rather than the postseismic correction terms.
2. The velocities are North America-fixed.
3. The values are in millimeters per year.

Why would fitting uncorrected postseismic velocities bias an interseismic locking model?


## 3. Define San Andreas coordinates

We approximate a short fault segment as a straight line with strike $lpha$, measured clockwise from north. The origin is a point on the fault.

We define positive fault-perpendicular distance toward the **left side** of the fault when looking along strike. For the northwest-striking San Andreas, this is approximately toward the Pacific side.


In [ ]:
EARTH_RADIUS_KM = 6371.0


def geographic_offsets_km(lon, lat, origin_lon, origin_lat):
    """Small-distance east and north offsets from an origin."""
    east = EARTH_RADIUS_KM * np.cos(np.deg2rad(origin_lat)) * np.deg2rad(np.asarray(lon) - origin_lon)
    north = EARTH_RADIUS_KM * np.deg2rad(np.asarray(lat) - origin_lat)
    return east, north


def rotate_to_fault_coordinates(table, origin_lon, origin_lat, strike_deg):
    """Add fault-parallel/perpendicular positions and velocities."""
    result = table.copy()
    east, north = geographic_offsets_km(result.lon, result.lat, origin_lon, origin_lat)
    strike = np.deg2rad(strike_deg)

    # Parallel is positive along strike. Perpendicular is positive to the left.
    result["x_parallel_km"] = east*np.sin(strike) + north*np.cos(strike)
    result["x_perp_km"] = -east*np.cos(strike) + north*np.sin(strike)
    result["v_parallel"] = result.ve*np.sin(strike) + result.vn*np.cos(strike)
    result["v_perp"] = -result.ve*np.cos(strike) + result.vn*np.sin(strike)
    result["s_parallel"] = np.sqrt(
        (result.se*np.sin(strike))**2 + (result.sn*np.cos(strike))**2
    )
    positive_sigma = result.loc[result.s_parallel > 0, "s_parallel"]
    if len(positive_sigma):
        result.loc[result.s_parallel <= 0, "s_parallel"] = positive_sigma.median()
    return result


def add_state_outlines(ax, state_codes=("CA", "NV")):
    try:
        from bokeh.sampledata.us_states import data as states
        for code in state_codes:
            ax.plot(states[code]["lons"], states[code]["lats"], color="0.4", lw=0.8, zorder=0)
    except Exception:
        pass


### Predict the transformation

For a station moving northwest approximately parallel to the San Andreas:

1. Will most of its motion appear in $v_{\parallel}$ or $v_{\perp}$?
2. Does rotating the axes change the station's actual velocity magnitude?
3. Why is a narrow along-strike sampling window important for a one-dimensional profile?


## 4. Guided example: the relatively locked Carrizo segment

The values below approximate a straight section of the San Andreas. The selection includes stations within a prescribed along-strike and cross-fault distance. Adjust the windows if the map reveals poor coverage.


In [ ]:
CARRIZO = {
    "origin_lon": -120.15,
    "origin_lat": 35.25,
    "strike_deg": 320.0,
    "along_halfwidth_km": 120.0,
    "cross_halfwidth_km": 180.0,
}

carrizo_all = rotate_to_fault_coordinates(
    velocity,
    CARRIZO["origin_lon"], CARRIZO["origin_lat"], CARRIZO["strike_deg"]
)
carrizo = carrizo_all[
    carrizo_all.x_parallel_km.abs().le(CARRIZO["along_halfwidth_km"])
    & carrizo_all.x_perp_km.abs().le(CARRIZO["cross_halfwidth_km"])
].copy()

print(f"Selected {len(carrizo)} stations")
display(carrizo[["station", "lon", "lat", "x_perp_km", "x_parallel_km", "v_parallel"]].head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
add_state_outlines(ax)
ax.scatter(carrizo_all.lon, carrizo_all.lat, s=8, color="0.8", label="other stations")
ax.scatter(carrizo.lon, carrizo.lat, s=25, color="#2166ac", label="profile selection")

# Plot the idealized straight fault segment through the origin.
strike = np.deg2rad(CARRIZO["strike_deg"])
lengths = np.array([-150, 150])
fault_e = lengths*np.sin(strike)
fault_n = lengths*np.cos(strike)
fault_lon = CARRIZO["origin_lon"] + np.rad2deg(fault_e/(EARTH_RADIUS_KM*np.cos(np.deg2rad(CARRIZO["origin_lat"]))))
fault_lat = CARRIZO["origin_lat"] + np.rad2deg(fault_n/EARTH_RADIUS_KM)
ax.plot(fault_lon, fault_lat, color="#b2182b", lw=2.5, label="idealized SAF")
ax.set(xlim=(-122.5, -118.0), ylim=(33.5, 37.2), xlabel="Longitude", ylabel="Latitude",
       title="Stations selected for the Carrizo profile")
ax.set_aspect(1/np.cos(np.deg2rad(35)))
ax.legend(frameon=False)

ax = axes[1]
ax.errorbar(carrizo.x_perp_km, carrizo.v_parallel, yerr=carrizo.s_parallel,
            fmt="o", ms=4, color="#2166ac", ecolor="0.65", alpha=0.85)
ax.axvline(0, color="#b2182b", ls="--", lw=1.5, label="idealized fault")
ax.set(xlabel="Fault-perpendicular distance (km; positive toward Pacific side)",
       ylabel="Fault-parallel velocity (mm/yr)", title="Observed fault-parallel velocity profile")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


### Inspect before fitting

1. Does the velocity profile resemble an arctangent?
2. What approximate far-field velocity difference do you see?
3. Are the two sides sampled equally well?
4. Identify any stations that appear inconsistent with the overall trend. Should they automatically be removed?


## 5. Fit the arctangent locking model

We fit

$$
v_{\parallel}(x)=c+\frac{V}{\pi}\tan^{-1}\left(\frac{x}{D}\right).
$$

The model parameters are the relative slip rate $V$, locking depth $D$, and velocity offset $c$.


In [ ]:
def arctan_velocity(x_km, slip_rate_mm_yr, locking_depth_km, offset_mm_yr):
    return offset_mm_yr + slip_rate_mm_yr/np.pi * np.arctan(x_km/locking_depth_km)


initial_guess = [35.0, 15.0, np.median(carrizo.v_parallel)]
bounds = ([0.0, 1.0, -100.0], [100.0, 80.0, 100.0])

parameters, covariance = curve_fit(
    arctan_velocity,
    carrizo.x_perp_km,
    carrizo.v_parallel,
    p0=initial_guess,
    sigma=carrizo.s_parallel,
    absolute_sigma=True,
    bounds=bounds,
)
parameter_sigma = np.sqrt(np.diag(covariance))
V_fit, D_fit, c_fit = parameters

results = pd.DataFrame({
    "estimate": parameters,
    "formal_sigma": parameter_sigma,
    "units": ["mm/yr", "km", "mm/yr"],
}, index=["slip rate V", "locking depth D", "velocity offset c"])
display(results)


In [ ]:
x_model = np.linspace(carrizo.x_perp_km.min(), carrizo.x_perp_km.max(), 600)
v_model = arctan_velocity(x_model, *parameters)
v_predicted = arctan_velocity(carrizo.x_perp_km, *parameters)
residual = carrizo.v_parallel - v_predicted

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]})
axes[0].errorbar(carrizo.x_perp_km, carrizo.v_parallel, yerr=carrizo.s_parallel,
                 fmt="o", ms=4, color="#2166ac", ecolor="0.7", label="GNSS")
axes[0].plot(x_model, v_model, color="#b2182b", lw=3, label="best-fitting arctangent")
axes[0].axvline(0, color="0.35", ls="--")
axes[0].set(ylabel="Fault-parallel velocity (mm/yr)", title="Carrizo locking model")
axes[0].legend(frameon=False)

axes[1].axhline(0, color="0.35", lw=1)
axes[1].scatter(carrizo.x_perp_km, residual, s=22, color="#2166ac")
axes[1].set(xlabel="Fault-perpendicular distance (km)", ylabel="Residual\n(mm/yr)")
plt.tight_layout()
plt.show()

print(f"RMS residual: {np.sqrt(np.mean(residual**2)):.2f} mm/yr")


### Interpret the model

1. Report the estimated slip rate and locking depth.
2. Which feature of the curve constrains slip rate? Which constrains locking depth?
3. Are the formal parameter uncertainties believable given the residual pattern?
4. Do residuals vary randomly, or are nearby stations systematically above or below the model?
5. Name two physical assumptions of the model that are violated by the real San Andreas system.


## 6. See the inverse problem before Week 3

The following grid holds the offset at its fitted value and calculates weighted misfit for combinations of slip rate and locking depth. A long, narrow minimum indicates a parameter tradeoff.


In [ ]:
V_grid = np.linspace(max(5, V_fit-20), V_fit+20, 100)
D_grid = np.linspace(2, min(60, D_fit+35), 100)
misfit = np.empty((len(D_grid), len(V_grid)))

for i, D_test in enumerate(D_grid):
    for j, V_test in enumerate(V_grid):
        prediction = arctan_velocity(carrizo.x_perp_km, V_test, D_test, c_fit)
        misfit[i, j] = np.sum(((carrizo.v_parallel - prediction)/carrizo.s_parallel)**2)

fig, ax = plt.subplots(figsize=(8, 6))
levels = np.nanpercentile(misfit, [1, 2, 5, 10, 20, 40, 70])
contour = ax.contour(V_grid, D_grid, misfit, levels=np.unique(levels), cmap="viridis")
ax.clabel(contour, inline=True, fontsize=8)
ax.plot(V_fit, D_fit, "r*", ms=14, label="best fit")
ax.set(xlabel="Slip rate V (mm/yr)", ylabel="Locking depth D (km)",
       title="Slip rate–locking depth misfit surface")
ax.legend(frameon=False)
plt.show()


### Interpret the tradeoff

1. Does the acceptable region form a compact circle or an elongated valley?
2. If you increase locking depth, how must slip rate change to retain a similar profile?
3. What additional observations could reduce this tradeoff?


## 7. Creep experiment: locked versus creeping behavior

We represent partial coupling as the sum of:

- a broad locked-fault contribution with fraction $C$;
- a narrow creeping contribution with fraction $1-C$.

The narrow transition is represented with `tanh` to avoid an ideal mathematical discontinuity.


In [ ]:
def coupled_velocity(x_km, slip_rate, locking_depth, offset, coupling, creep_width_km=1.0):
    locked = coupling * slip_rate/np.pi * np.arctan(x_km/locking_depth)
    creeping = (1-coupling) * slip_rate/2 * np.tanh(x_km/creep_width_km)
    return offset + locked + creeping


fig, ax = plt.subplots(figsize=(9, 5.5))
for coupling, color in zip([1.0, 0.5, 0.0], ["#2166ac", "#7b3294", "#b2182b"]):
    ax.plot(x_model, coupled_velocity(x_model, V_fit, D_fit, c_fit, coupling),
            lw=3, color=color, label=f"C = {coupling:g}")
ax.set(xlim=(-80, 80), xlabel="Fault-perpendicular distance (km)",
       ylabel="Fault-parallel velocity (mm/yr)",
       title="The same far-field rate can be locked or creeping")
ax.axvline(0, color="0.3", ls="--")
ax.legend(frameon=False)
plt.show()


### Creep interpretation

1. Which curve stores the greatest elastic strain near the fault?
2. Which curve has the largest velocity change immediately at the fault?
3. Do all curves accommodate the same long-term far-field motion?
4. Where would you place GNSS stations to distinguish partial coupling from full locking?
5. Would a station spacing of 50 km resolve a 1-km-wide creeping zone?


## 8. Optional application: Parkfield

Parkfield lies near the transition between creeping and more strongly locked portions of the San Andreas. Repeat the coordinate transformation and plot its profile. Do not assume that the fully locked arctangent model will fit.


In [ ]:
PARKFIELD = {
    "origin_lon": -120.50,
    "origin_lat": 35.90,
    "strike_deg": 320.0,
    "along_halfwidth_km": 80.0,
    "cross_halfwidth_km": 120.0,
}

parkfield_all = rotate_to_fault_coordinates(
    velocity,
    PARKFIELD["origin_lon"], PARKFIELD["origin_lat"], PARKFIELD["strike_deg"]
)
parkfield = parkfield_all[
    parkfield_all.x_parallel_km.abs().le(PARKFIELD["along_halfwidth_km"])
    & parkfield_all.x_perp_km.abs().le(PARKFIELD["cross_halfwidth_km"])
].copy()

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.errorbar(parkfield.x_perp_km, parkfield.v_parallel, yerr=parkfield.s_parallel,
            fmt="o", ms=4, color="#2166ac", ecolor="0.7")
ax.axvline(0, color="#b2182b", ls="--", label="idealized fault")
ax.set(xlabel="Fault-perpendicular distance (km)", ylabel="Fault-parallel velocity (mm/yr)",
       title="Parkfield-region velocity profile")
ax.legend(frameon=False)
plt.show()


### Compare Carrizo and Parkfield

1. Does Parkfield show a sharper near-fault velocity change?
2. Would changing only locking depth reproduce that shape?
3. What observations would demonstrate surface creep directly?
4. Why is coupling likely to vary along strike rather than having one value for the entire San Andreas?


## 9. From slip deficit to an earthquake moment budget

Assume the fitted Carrizo slip rate is the long-term rate and choose a coupling fraction $C$ and recurrence interval $T$:

$$
S_{deficit}=CVT.
$$

For rupture length $L$, width $W$, and shear modulus $\mu$:

$$
M_0=\mu LWS,
\qquad
M_w=\frac{2}{3}\left(\log_{10}M_0-9.1\right).
$$


In [ ]:
coupling = 1.0
recurrence_years = np.array([50, 100, 150, 250])
rupture_length_km = 300.0
rupture_width_km = 15.0
shear_modulus_pa = 30e9

slip_deficit_m = coupling * (V_fit * 1e-3) * recurrence_years
area_m2 = rupture_length_km*1e3 * rupture_width_km*1e3
moment_nm = shear_modulus_pa * area_m2 * slip_deficit_m
mw = (2/3) * (np.log10(moment_nm) - 9.1)

budget = pd.DataFrame({
    "recurrence_yr": recurrence_years,
    "slip_deficit_m": slip_deficit_m,
    "moment_Nm": moment_nm,
    "Mw_equivalent": mw,
})
display(budget)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(recurrence_years, slip_deficit_m, "o-", color="#2166ac", lw=2.5)
axes[0].set(xlabel="Recurrence interval (yr)", ylabel="Slip deficit (m)",
            title="Accumulated slip deficit")
axes[1].plot(recurrence_years, mw, "o-", color="#b2182b", lw=2.5)
axes[1].set(xlabel="Recurrence interval (yr)", ylabel="$M_w$ equivalent",
            title="Moment-budget equivalent")
plt.tight_layout()
plt.show()


### Interpret the earthquake budget

1. How much slip deficit accumulates over 150 years?
2. Why does doubling recurrence interval not double moment magnitude?
3. Recalculate with $C=0.5$. What changes?
4. List at least three assumptions required to interpret this as one earthquake.
5. Explain why the result is a moment-budget equivalent and not a forecast.


## 10. Synthesis

Write a short paragraph answering the following:

> What aspects of the observed San Andreas velocity profile can be explained by a single locked-fault model, and what aspects require additional processes or structures?

Your response should mention coordinate rotation, locking depth, slip rate, creep or coupling, residuals, and at least one limitation of the moment-budget calculation.

### References

- Kreemer, C., Hammond, W. C., and Blewitt, G. (2022), *Crustal Strain Rates in the Western United States and Their Relationship with Earthquake Rates*.
- Savage, J. C. and Burford, R. O. (1973), Geodetic determination of relative plate motion in central California.
- Reid, H. F. (1910), *The Mechanics of the Earthquake*.
